# Bluestock Mutual Fund Analytics

## 02 - Data Cleaning

This notebook performs data cleaning and validation on the mutual fund
datasets. The process includes duplicate removal, missing-value analysis,
data-type corrections, date standardisation, and basic data-quality checks.

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: c:\Users\Farhan\bluestock_mf_capstone\data\raw
Processed folder: c:\Users\Farhan\bluestock_mf_capstone\data\processed


In [13]:
csv_files = sorted(
    [
        f for f in RAW_DIR.glob("*.csv")
        if not f.name.endswith("_live_nav.csv")
    ]
)

datasets = {}

for file in csv_files:
    df = pd.read_csv(file)
    datasets[file.stem] = df

print("Datasets loaded:", len(datasets))

Datasets loaded: 10


In [14]:
quality_report = []

for name, df in datasets.items():
    quality_report.append({
        "dataset": name,
        "rows_before": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_columns": int((df.isna().sum() > 0).sum())
    })

quality_df = pd.DataFrame(quality_report)

display(
    quality_df.sort_values(
        "missing_cells",
        ascending=False
    )
)

,dataset,rows_before,columns,duplicate_rows,missing_cells,missing_columns
5,1788499984405-d702a6c6-04_monthly_sip_inflows,48,6,0,12,1
0,1788499980509-304c1255-08_investor_transactions,32778,13,0,0,0
1,1788499982117-e3d6ab98-09_portfolio_holdings,322,8,0,0,0
2,1788499982615-f9647ab2-10_benchmark_indices,8050,3,0,0,0
3,1788499983331-4389156d-02_nav_history,46000,3,0,0,0
4,1788499984134-b0cbf625-03_aum_by_fund_house,90,5,0,0,0
6,1788499984721-4b860901-05_category_inflows,144,3,0,0,0
7,1788499985036-da4a0c4a-06_industry_folio_count,21,6,0,0,0
8,1788499985420-bb134abf-07_scheme_performance,40,19,0,0,0
9,fund_master,40,15,0,0,0


In [15]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print("\n" + "=" * 70)
        print(name)
        print("=" * 70)
        display(missing.to_frame("missing_count"))


1788499984405-d702a6c6-04_monthly_sip_inflows


,missing_count
yoy_growth_pct,12


In [16]:
for name, df in datasets.items():
    duplicates = df.duplicated().sum()

    print(
        f"{name}: "
        f"{duplicates} duplicate rows"
    )

1788499980509-304c1255-08_investor_transactions: 0 duplicate rows
1788499982117-e3d6ab98-09_portfolio_holdings: 0 duplicate rows
1788499982615-f9647ab2-10_benchmark_indices: 0 duplicate rows
1788499983331-4389156d-02_nav_history: 0 duplicate rows
1788499984134-b0cbf625-03_aum_by_fund_house: 0 duplicate rows
1788499984405-d702a6c6-04_monthly_sip_inflows: 0 duplicate rows
1788499984721-4b860901-05_category_inflows: 0 duplicate rows
1788499985036-da4a0c4a-06_industry_folio_count: 0 duplicate rows
1788499985420-bb134abf-07_scheme_performance: 0 duplicate rows
fund_master: 0 duplicate rows


In [17]:
cleaned_datasets = {}

for name, df in datasets.items():

    cleaned_df = df.copy()

    # Remove completely empty rows and columns
    cleaned_df = cleaned_df.dropna(
        axis=0,
        how="all"
    )

    cleaned_df = cleaned_df.dropna(
        axis=1,
        how="all"
    )

    # Remove exact duplicate rows
    cleaned_df = cleaned_df.drop_duplicates()

    # Standardise column names
    cleaned_df.columns = (
        cleaned_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # Remove unnecessary spaces from text columns
    text_columns = cleaned_df.select_dtypes(
        include="object"
    ).columns

    for col in text_columns:
        cleaned_df[col] = cleaned_df[col].str.strip()

    cleaned_datasets[name] = cleaned_df

print("Basic cleaning completed.")

Basic cleaning completed.


C:\Users\Farhan\AppData\Local\Temp\ipykernel_11308\3874992338.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = cleaned_df.select_dtypes(
C:\Users\Farhan\AppData\Local\Temp\ipykernel_11308\3874992338.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-

In [18]:
cleaned_datasets = {}

for name, df in datasets.items():

    cleaned_df = df.copy()

    # Remove completely empty rows and columns
    cleaned_df = cleaned_df.dropna(
        axis=0,
        how="all"
    )

    cleaned_df = cleaned_df.dropna(
        axis=1,
        how="all"
    )

    # Remove exact duplicate rows
    cleaned_df = cleaned_df.drop_duplicates()

    # Standardise column names
    cleaned_df.columns = (
        cleaned_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # Remove unnecessary spaces from text columns
    text_columns = cleaned_df.select_dtypes(
        include="object"
    ).columns

    for col in text_columns:
        cleaned_df[col] = cleaned_df[col].str.strip()

    cleaned_datasets[name] = cleaned_df

print("Basic cleaning completed.")


Basic cleaning completed.


C:\Users\Farhan\AppData\Local\Temp\ipykernel_11308\516482977.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = cleaned_df.select_dtypes(
C:\Users\Farhan\AppData\Local\Temp\ipykernel_11308\516482977.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-

In [19]:
for name, df in cleaned_datasets.items():

    for col in df.columns:

        col_lower = col.lower()

        if (
            "date" in col_lower
            or col_lower == "month"
        ):
            try:
                df[col] = pd.to_datetime(
                    df[col],
                    errors="coerce"
                )
            except Exception:
                pass

In [20]:
numeric_summary = []

for name, df in cleaned_datasets.items():

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    numeric_summary.append({
        "dataset": name,
        "numeric_columns": len(numeric_cols),
        "numeric_missing_values": int(
            df[numeric_cols].isna().sum().sum()
        )
    })

numeric_summary_df = pd.DataFrame(
    numeric_summary
)

display(numeric_summary_df)

,dataset,numeric_columns,numeric_missing_values
0,1788499980509-304c1255-08_investor_transactions,3,0
1,1788499982117-e3d6ab98-09_portfolio_holdings,4,0
2,1788499982615-f9647ab2-10_benchmark_indices,1,0
3,1788499983331-4389156d-02_nav_history,2,0
4,1788499984134-b0cbf625-03_aum_by_fund_house,3,0
5,1788499984405-d702a6c6-04_monthly_sip_inflows,5,12
6,1788499984721-4b860901-05_category_inflows,1,0
7,1788499985036-da4a0c4a-06_industry_folio_count,5,0
8,1788499985420-bb134abf-07_scheme_performance,14,0
9,fund_master,5,0


In [21]:
for name, df in cleaned_datasets.items():

    output_file = PROCESSED_DIR / f"{name}.csv"

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved: {output_file.name} "
        f"| Shape: {df.shape}"
    )

Saved: 1788499980509-304c1255-08_investor_transactions.csv | Shape: (32778, 13)
Saved: 1788499982117-e3d6ab98-09_portfolio_holdings.csv | Shape: (322, 8)
Saved: 1788499982615-f9647ab2-10_benchmark_indices.csv | Shape: (8050, 3)
Saved: 1788499983331-4389156d-02_nav_history.csv | Shape: (46000, 3)
Saved: 1788499984134-b0cbf625-03_aum_by_fund_house.csv | Shape: (90, 5)
Saved: 1788499984405-d702a6c6-04_monthly_sip_inflows.csv | Shape: (48, 6)
Saved: 1788499984721-4b860901-05_category_inflows.csv | Shape: (144, 3)
Saved: 1788499985036-da4a0c4a-06_industry_folio_count.csv | Shape: (21, 6)
Saved: 1788499985420-bb134abf-07_scheme_performance.csv | Shape: (40, 19)
Saved: fund_master.csv | Shape: (40, 15)


In [22]:
final_report = []

for name, df in cleaned_datasets.items():

    final_report.append({
        "dataset": name,
        "rows_after": len(df),
        "columns_after": len(df.columns),
        "duplicate_rows_after": int(
            df.duplicated().sum()
        ),
        "missing_cells_after": int(
            df.isna().sum().sum()
        )
    })

final_report_df = pd.DataFrame(final_report)

display(final_report_df)

,dataset,rows_after,columns_after,duplicate_rows_after,missing_cells_after
0,1788499980509-304c1255-08_investor_transactions,32778,13,0,0
1,1788499982117-e3d6ab98-09_portfolio_holdings,322,8,0,0
2,1788499982615-f9647ab2-10_benchmark_indices,8050,3,0,0
3,1788499983331-4389156d-02_nav_history,46000,3,0,0
4,1788499984134-b0cbf625-03_aum_by_fund_house,90,5,0,0
5,1788499984405-d702a6c6-04_monthly_sip_inflows,48,6,0,12
6,1788499984721-4b860901-05_category_inflows,144,3,0,0
7,1788499985036-da4a0c4a-06_industry_folio_count,21,6,0,0
8,1788499985420-bb134abf-07_scheme_performance,40,19,0,0
9,fund_master,40,15,0,0


## Data Cleaning Summary

- All 10 original datasets were loaded successfully.
- Completely empty rows and columns were removed.
- Exact duplicate rows were removed where present.
- Column names were standardised to lowercase snake_case.
- Leading and trailing whitespace was removed from text fields.
- Date-like columns were converted to standard datetime format where possible.
- Missing values were identified and retained where they represent unavailable source information rather than being replaced blindly.
- Cleaned datasets were saved to the `data/processed` directory.
- Final validation was performed after cleaning.